# Station-level MLP: multi-output regression

This notebook predicts two targets jointly:

1. `inflow_count`
2. `outflow_count`

It deliberately loads the **same prepared dataframe** and **same saved spatial fold assignments** as CatBoost. Dataset-construction functions remain in the CatBoost preparation notebook. Only model-independent loading, fold reconstruction, target transforms, metrics, reproducibility, preprocessing, and neural components are imported from `model_helper_stations_lvl.py`. MiniLM embedding creation stays in this notebook, is saved once, and can later be reused by GAT.

Canonical target order throughout the notebook: **inflow first, outflow second**.


In [1]:
# Import os.
import os
# Import sys.
import sys
# Import from copy.
from copy import deepcopy
# Import from pathlib.
from pathlib import Path

# Import numpy.
import numpy as np
# Import pandas.
import pandas as pd
# Import torch.
import torch
# Import torch.nn.
import torch.nn as nn
# Import from torch.utils.data.
from torch.utils.data import DataLoader, TensorDataset

# Import from sklearn.decomposition.
from sklearn.decomposition import PCA
# Import from sklearn.model_selection.
from sklearn.model_selection import ParameterGrid

# Import the MiniLM sentence encoder.
from sentence_transformers import SentenceTransformer

# Import from IPython.display.
from IPython.display import display

# Set the project path.
project_root = Path("/home/najla/dev/najla-msc/bikeshare")
# Add the project path if needed.
if str(project_root) not in sys.path:
    # Run append.
    sys.path.append(str(project_root))

# Import from bikeshare_scripts.model_helper_stations_lvl.
from scripts.model_helper_stations_lvl import (
    BikeShareModelHelper as mh,
    FeatureEncoder,
    RegressionHead,
    get_activation,
    maybe_layer_norm,
)


In [2]:
# Choose CPU or GPU.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Set the random seed.
mh.set_random_seed(42)
# Print the result.
print("PyTorch device:", device)


PyTorch device: cuda


## 1. Load the exact CatBoost modelling dataframe and SPCV design

No dataset-generation function is repeated here. The two parquet files are the shared source of truth.


In [3]:
# Set the data folder.
model_df_dir = (
    project_root
    / "data/processed/model_df"
)

# Set the model-data file.
model_df_file = (
    model_df_dir
    / "df_2targets_txtTokens_graphFeat.parquet"
)

# Set the fold file.
fold_assignments_file = (
    model_df_dir
    / "folds_of_2targets_txtTokens_graphFeat.parquet"
)

# Set df, fold assignments df.
df, fold_assignments_df = mh.load_station_level_datasets(
    model_df_file=model_df_file,
    fold_assignments_file=fold_assignments_file,
)

# Print the result.
print("Model dataframe shape:", df.shape)
# Print the result.
print("Fold-assignment shape:", fold_assignments_df.shape)
# Show the result.
display(df.head())
# Show the result.
display(fold_assignments_df.head(20))


Model dataframe shape: (740505, 73)
Fold-assignment shape: (200, 3)


,loc_id,loc_name,lon,lat,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,...,node2vec_12,node2vec_13,node2vec_14,node2vec_15,day_type,public_holiday,Main_Weather_Category,season,spatial_group,attraction_missing_flag
0,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,new_years_day,snowy,winter,19,0
1,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,new_years_day,snowy,winter,19,0
2,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,new_years_day,snowy,winter,19,0
3,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,new_years_day,snowy,winter,19,0
4,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,new_years_day,snowy,winter,19,0


,fold_id,spatial_group,split
0,0,0,test
1,0,5,test
2,0,19,test
3,0,1,train
4,0,2,train
5,0,3,train
6,0,7,train
7,0,8,train
8,0,9,train
9,0,10,train


## 2. Create MiniLM embeddings from the full Wikidata table

MiniLM reads the complete `all_wikidata_tokens.parquet` file from the graph folder. The embedding table is saved once in the same folder, joined to the MLP dataframe by `loc_id`, and the original `wiki_items_text` column is then removed.


In [6]:
# Set the graph folder.
graph_dir = (
    project_root
    / "data/processed/graph"
)

# Set the full Wikidata-token file.
wikidata_tokens_file = (
    graph_dir
    / "cat_all_wikidata_tokens.parquet"
) # chnage bac to "all_wikidata_tokens.parquet" if testing failed

# Set the saved MiniLM file.
embedding_cache_file = (
    graph_dir
    / "cat_all_wikidata_minilm_embeddings.parquet"
)

# Load the full Wikidata-token table.
wikidata_tokens_df = pd.read_parquet(
    wikidata_tokens_file
)

# Prepare the text for MiniLM.
wikidata_tokens_df["wiki_items_text"] = (
    wikidata_tokens_df["wiki_items_text"]
    .fillna("no_wikidata")
    .astype(str)
    )

# Load MiniLM.
text_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
    )

# Encode every CT in the full Wikidata table.
wiki_embeddings = text_model.encode(
    wikidata_tokens_df["wiki_items_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    )

# Name the embedding columns.
wiki_cols = [
    f"wiki_emb_{i}"
    for i in range(wiki_embeddings.shape[1])
    ]

# Create the saved embedding table.
wiki_embedding_df = pd.concat(
        [
            wikidata_tokens_df[["loc_id"]].reset_index(drop=True),
            pd.DataFrame(
                wiki_embeddings,
                columns=wiki_cols,
            ),
        ],
        axis=1,
    )

# Save the embeddings in the graph folder.
wiki_embedding_df.to_parquet(
        embedding_cache_file,
        index=False,
    )

# Get the embedding-column names.
wiki_cols = [
    col
    for col in wiki_embedding_df.columns
    if col.startswith("wiki_emb_")
]

# Join the MiniLM embeddings to the MLP dataframe.
df = df.merge(
    wiki_embedding_df,
    on="loc_id",
    how="left",
)

# Remove the original text column.
df = df.drop(
    columns=["wiki_items_text"]
)

# Print the saved file.
print("MiniLM file:", embedding_cache_file)
# Print the embedding-table shape.
print("Full MiniLM table shape:", wiki_embedding_df.shape)
# Print the updated MLP dataframe shape.
print("MLP dataframe shape:", df.shape)
# Show the joined embeddings.
display(df[["loc_id"] + wiki_cols[:5]].head())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

MiniLM file: /home/najla/dev/najla-msc/bikeshare/data/processed/graph/cat_all_wikidata_minilm_embeddings.parquet
Full MiniLM table shape: (469, 385)
MLP dataframe shape: (740505, 456)


,loc_id,wiki_emb_0,wiki_emb_1,wiki_emb_2,wiki_emb_3,wiki_emb_4
0,5350001.00,0.122401,-0.04867,0.080268,0.035624,-0.014239
1,5350001.00,0.122401,-0.04867,0.080268,0.035624,-0.014239
2,5350001.00,0.122401,-0.04867,0.080268,0.035624,-0.014239
3,5350001.00,0.122401,-0.04867,0.080268,0.035624,-0.014239
4,5350001.00,0.122401,-0.04867,0.080268,0.035624,-0.014239


In [ ]:
error
df.head()

,loc_id,loc_name,lon,lat,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,...,wiki_emb_374,wiki_emb_375,wiki_emb_376,wiki_emb_377,wiki_emb_378,wiki_emb_379,wiki_emb_380,wiki_emb_381,wiki_emb_382,wiki_emb_383
0,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,0.067846,0.062455,0.005089,0.068023,-0.007927,0.019729,0.027136,0.107046,-0.004424,0.00981
1,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,0.067846,0.062455,0.005089,0.068023,-0.007927,0.019729,0.027136,0.107046,-0.004424,0.00981
2,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,0.067846,0.062455,0.005089,0.068023,-0.007927,0.019729,0.027136,0.107046,-0.004424,0.00981
3,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,0.067846,0.062455,0.005089,0.068023,-0.007927,0.019729,0.027136,0.107046,-0.004424,0.00981
4,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,0.067846,0.062455,0.005089,0.068023,-0.007927,0.019729,0.027136,0.107046,-0.004424,0.00981


## 3. Define the same targets and non-text feature scope as CatBoost


In [ ]:
# Target order: inflow, then outflow.
target_cols = [
    "inflow_count",
    "outflow_count",
]

# Set the date column.
date_col = "date"
# Set the ID column.
id_col = "station_name"
# Set the spatial-group column.
group_col = "spatial_group"

# List the categorical features.
categorical_cols = [
    "day_type",
    "public_holiday",
    "Main_Weather_Category",
    "season",
]

# Match the CatBoost feature exclusions.
# MLP uses MiniLM instead of raw text.
drop_feature_cols = [
    "loc_id",
    "loc_name",
    "loc_id_key",
    "lon",
    "lat",
    "spatial_group",
    "date",
    "inflow_count",
    "outflow_count",
    "end_station_name",
    "start_station_name",
    "station_name",
    "start_station_count",
    "end_station_count",
    "start_capacity_avg",
]

# List the excluded features.
drop_feature_cols = [
    col for col in drop_feature_cols
    if col in df.columns
]

# Find the numerical features.
numerical_cols = [
    col
    for col in df.columns
    if (
        col not in drop_feature_cols
        and col not in categorical_cols
        and col not in wiki_cols
    )
]


# Print the result.
print("Targets:", target_cols)
# Print the result.
print("Categorical feature count:", len(categorical_cols))
# Print the result.
print("Numerical feature count:", len(numerical_cols))
# Print the result.
print("MiniLM feature count:", len(wiki_cols))


Targets: ['inflow_count', 'outflow_count']
Categorical feature count: 4
Numerical feature count: 59
MiniLM feature count: 384


## 4. Reconstruct the exact saved spatial folds


In [ ]:
# Set spcv folds.
spcv_folds = mh.prepare_spatial_folds_from_assignments(
    df=df,
    fold_assignments_df=fold_assignments_df,
    group_col=group_col,
    include_dataframes=False,
)

# Print the result.
print("Number of SPCV folds:", len(spcv_folds))

# Loop through fold.
for fold in spcv_folds:
    # Print the result.
    print(
        f"Fold {fold['fold_id']:02d} | "
        f"train groups={fold['train_groups']} | "
        f"val groups={fold['val_groups']} | "
        f"test groups={fold['test_groups']}"
    )


Number of SPCV folds: 10
Fold 00 | train groups=(0, 3, 4, 5, 6, 8, 9, 10, 12, 13, 14, 15, 17, 18) | val groups=(1, 7, 11) | test groups=(2, 16, 19)
Fold 01 | train groups=(2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 18) | val groups=(0, 1, 5) | test groups=(13, 17, 19)
Fold 02 | train groups=(0, 3, 5, 6, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19) | val groups=(1, 4, 7) | test groups=(2, 8, 12)
Fold 03 | train groups=(0, 3, 4, 5, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19) | val groups=(1, 6, 7) | test groups=(2, 8, 12)
Fold 04 | train groups=(0, 3, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18) | val groups=(2, 16, 19) | test groups=(1, 4, 5)
Fold 05 | train groups=(0, 2, 3, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 18) | val groups=(1, 4, 7) | test groups=(16, 17, 19)
Fold 06 | train groups=(0, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14, 15, 18) | val groups=(16, 17, 19) | test groups=(1, 10, 11)
Fold 07 | train groups=(0, 3, 4, 6, 7, 8, 9, 10, 11, 13, 14, 15, 17, 18) | val groups=(2, 16, 19) | test groups=(1

## 5. Prepare each MLP fold

The split itself is shared. PCA, scaling, and one-hot encoding are fitted **only on the training rows of each fold** to prevent leakage.


In [ ]:
# Define prepare_mlp_fold.
def prepare_mlp_fold(
    df,
    fold,
    numerical_cols,
    categorical_cols,
    wiki_cols,
    *,
    n_wiki_components=16,
):
    """Prepare one saved SPCV fold for multi-output MLP training."""

    # Get training rows.
    train_df = df.iloc[fold["train_idx"]].copy()
    # Get validation rows.
    val_df = df.iloc[fold["val_idx"]].copy()
    # Get test rows.
    test_df = df.iloc[fold["test_idx"]].copy()

    # Check the condition.
    if len(wiki_cols) < n_wiki_components:
        # Stop with an error.
        raise ValueError(
            "n_wiki_components cannot exceed the MiniLM dimension."
        )

    # Set pca.
    pca = PCA(
        n_components=n_wiki_components,
        random_state=42,
    )

    # Set wiki train.
    wiki_train = pca.fit_transform(
        train_df[wiki_cols].fillna(0)
    )
    # Apply the already-fitted transformer and store the result as `wiki_val`.
    wiki_val = pca.transform(
        val_df[wiki_cols].fillna(0)
    )
    # Apply the already-fitted transformer and store the result as `wiki_test`.
    wiki_test = pca.transform(
        test_df[wiki_cols].fillna(0)
    )

    # Set wiki pca cols.
    wiki_pca_cols = [
        f"wiki_pca_{i}"
        for i in range(n_wiki_components)
    ]

    # Get training rows.
    train_df[wiki_pca_cols] = wiki_train
    # Get validation rows.
    val_df[wiki_pca_cols] = wiki_val
    # Get test rows.
    test_df[wiki_pca_cols] = wiki_test

    # Set fold numerical cols.
    fold_numerical_cols = [
        *numerical_cols,
        *wiki_pca_cols,
    ]

    # Set preprocessor.
    preprocessor = mh.build_feature_preprocessor(
        numerical_cols=fold_numerical_cols,
        categorical_cols=categorical_cols,
    )

    # Prepare training features.
    X_train = np.asarray(
        preprocessor.fit_transform(train_df),
        dtype=np.float32,
    )
    # Prepare validation features.
    X_val = np.asarray(
        preprocessor.transform(val_df),
        dtype=np.float32,
    )
    # Prepare test features.
    X_test = np.asarray(
        preprocessor.transform(test_df),
        dtype=np.float32,
    )

    # Set y train raw.
    y_train_raw = train_df[target_cols].to_numpy(dtype=float)
    # Set y val raw.
    y_val_raw = val_df[target_cols].to_numpy(dtype=float)
    # Set y test raw.
    y_test_raw = test_df[target_cols].to_numpy(dtype=float)

    # Prepare training targets.
    y_train = mh.target_transform(
        y_train_raw,
        transform="log1p",
    ).astype(np.float32)
    # Prepare validation targets.
    y_val = mh.target_transform(
        y_val_raw,
        transform="log1p",
    ).astype(np.float32)
    # Prepare test targets.
    y_test = mh.target_transform(
        y_test_raw,
        transform="log1p",
    ).astype(np.float32)

    # Set test metadata.
    test_metadata = test_df[
        [
            date_col,
            id_col,
            group_col,
            *target_cols,
        ]
    ].copy().reset_index(drop=True)

    # Return the result.
    return {
        "fold_id": fold["fold_id"],
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "y_test_raw": y_test_raw,
        "input_dim": X_train.shape[1],
        "output_dim": len(target_cols),
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "test_metadata": test_metadata,
        "preprocessor": preprocessor,
        "pca": pca,
        "train_groups": fold["train_groups"],
        "val_groups": fold["val_groups"],
        "test_groups": fold["test_groups"],
    }


In [ ]:
# Set prepared folds.
prepared_folds = [
    prepare_mlp_fold(
        df=df,
        fold=fold,
        numerical_cols=numerical_cols,
        categorical_cols=categorical_cols,
        wiki_cols=wiki_cols,
        n_wiki_components=16,
    )
    for fold in spcv_folds
]

# Print the result.
print("Prepared MLP folds:", len(prepared_folds))
# Print the result.
print("First fold X_train shape:", prepared_folds[0]["X_train"].shape)
# Print the result.
print("First fold y_train shape:", prepared_folds[0]["y_train"].shape)


## 6. Multi-output DataLoaders and model


In [ ]:
# Define create_mlp_dataloaders.
def create_mlp_dataloaders(
    fold,
    *,
    batch_size=256,
):
    # Create the training dataset.
    train_dataset = TensorDataset(
        torch.tensor(fold["X_train"], dtype=torch.float32),
        torch.tensor(fold["y_train"], dtype=torch.float32),
    )
    # Create the validation dataset.
    val_dataset = TensorDataset(
        torch.tensor(fold["X_val"], dtype=torch.float32),
        torch.tensor(fold["y_val"], dtype=torch.float32),
    )
    # Create the test dataset.
    test_dataset = TensorDataset(
        torch.tensor(fold["X_test"], dtype=torch.float32),
        torch.tensor(fold["y_test"], dtype=torch.float32),
    )

    # Return the result.
    return (
        DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
        ),
        DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
        ),
        DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
        ),
    )


In [ ]:
# Define MLPRegressor.
class MLPRegressor(nn.Module):
    """Jointly predict log-inflow and log-outflow."""

    # Define __init__.
    def __init__(
        self,
        input_dim,
        hidden_dims,
        activation,
        use_layer_norm,
        dropout,
        output_dim=2,
    ):
        # Initialise the parent PyTorch module so parameters and submodules are registered correctly.
        super().__init__()

        # Check the condition.
        if not hidden_dims:
            # Stop with an error.
            raise ValueError("hidden_dims must contain at least one layer.")

        # Set encoder.
        self.encoder = FeatureEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dims[0],
            activation=activation,
            use_layer_norm=use_layer_norm,
            dropout=dropout,
        )

        # Set layers.
        layers = []
        # Loop through in dim, out dim.
        for in_dim, out_dim in zip(
            hidden_dims[:-1],
            hidden_dims[1:],
        ):
            # Extend `layers` with all supplied items.
            layers.extend(
                [
                    nn.Linear(in_dim, out_dim),
                    get_activation(activation),
                    maybe_layer_norm(
                        out_dim,
                        use_layer_norm,
                    ),
                    nn.Dropout(dropout),
                ]
            )

        # Set backbone.
        self.backbone = nn.Sequential(*layers)
        # Set head.
        self.head = RegressionHead(
            hidden_dim=hidden_dims[-1],
            output_dim=output_dim,
        )

    # Define forward.
    def forward(self, x):
        # Set x.
        x = self.encoder(x)
        # Set x.
        x = self.backbone(x)
        # Return the result.
        return self.head(x)


## 7. Hyperparameter grid


In [ ]:
# Define get_mlp_grid_exp.
def get_mlp_grid_exp():
    # Set adam grid.
    adam_grid = {
        "hidden_dims": [
            (128, 64),
            (128, 64, 32),
        ],
        "activation": [
            "relu",
            "gelu",
            "leaky_relu",
        ],
        "use_layer_norm": [True],
        "dropout": [0.20, 0.60],
        "optimizer_name": ["Adam"],
        "learning_rate": [0.001, 0.0005],
        "weight_decay": [0.0001, 0.001],
        "loss_name": ["MSE"],
        "batch_size": [64, 256],
        "max_epochs": [50, 300],
        "early_stopping_patience": [25],
        "scheduler_name": ["ReduceLROnPlateau"],
        "scheduler_patience": [8],
        "scheduler_factor": [0.50],
        "random_seed": [42],
    }

    # Set sgd grid.
    sgd_grid = {
        "hidden_dims": [
            (128, 64),
            (128, 64, 32),
        ],
        "activation": [
            "relu",
            "gelu",
            "leaky_relu",
        ],
        "use_layer_norm": [True],
        "dropout": [0.20, 0.60],
        "optimizer_name": ["SGD"],
        "learning_rate": [0.001, 0.0005],
        "momentum": [0.90],
        "weight_decay": [0.0001, 0.001],
        "loss_name": ["MSE"],
        "batch_size": [64, 256],
        "max_epochs": [50, 300],
        "early_stopping_patience": [25],
        "scheduler_name": ["ExponentialLR"],
        "scheduler_gamma": [0.95],
        "random_seed": [42],
    }

    # Return the result.
    return [adam_grid, sgd_grid]

# Set parameter combinations.
parameter_combinations = list(
    ParameterGrid(get_mlp_grid_exp())
)

# Print the result.
print(
    "Number of hyperparameter combinations:",
    len(parameter_combinations),
)


## 8. Train one fold and evaluate with the same CatBoost metrics


In [ ]:
# Define train_one_mlp_fold.
def train_one_mlp_fold(
    fold,
    params,
    *,
    device,
):
    # Set the random seed.
    mh.set_random_seed(
        params["random_seed"] + fold["fold_id"]
    )

    # Set train loader, val loader, test loader.
    train_loader, val_loader, test_loader = (
        create_mlp_dataloaders(
            fold=fold,
            batch_size=params["batch_size"],
        )
    )

    # Create the model.
    model = MLPRegressor(
        input_dim=fold["input_dim"],
        hidden_dims=params["hidden_dims"],
        activation=params["activation"],
        use_layer_norm=params["use_layer_norm"],
        dropout=params["dropout"],
        output_dim=fold["output_dim"],
    ).to(device)

    # Check the condition.
    if params["loss_name"] != "MSE":
        # Stop with an error.
        raise ValueError(
            f"Unknown loss: {params['loss_name']}"
        )
    # Set the loss function.
    criterion = nn.MSELoss()

    # Check the condition.
    if params["optimizer_name"] == "Adam":
        # Create the optimizer.
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=params["learning_rate"],
            weight_decay=params["weight_decay"],
        )
    # Test the next mutually exclusive condition: `params["optimizer_name"] == "SGD"`.
    elif params["optimizer_name"] == "SGD":
        # Create the optimizer.
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=params["learning_rate"],
            momentum=params["momentum"],
            weight_decay=params["weight_decay"],
        )
    else:
        # Stop with an error.
        raise ValueError(
            f"Unknown optimizer: {params['optimizer_name']}"
        )

    # Check the condition.
    if params["scheduler_name"] == "ReduceLROnPlateau":
        # Set scheduler.
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=params["scheduler_factor"],
            patience=params["scheduler_patience"],
        )
    # Test the next mutually exclusive condition: `params["scheduler_name"] == "ExponentialLR"`.
    elif params["scheduler_name"] == "ExponentialLR":
        # Set scheduler.
        scheduler = torch.optim.lr_scheduler.ExponentialLR(
            optimizer,
            gamma=params["scheduler_gamma"],
        )
    else:
        # Stop with an error.
        raise ValueError(
            f"Unknown scheduler: {params['scheduler_name']}"
        )

    # Store the best validation loss.
    best_val_loss = float("inf")
    # Store the best epoch.
    best_epoch = 0
    # Set best model state.
    best_model_state = None
    # Set epochs without improvement.
    epochs_without_improvement = 0

    # Loop through epoch.
    for epoch in range(1, params["max_epochs"] + 1):
        # Switch `model` to training mode.
        model.train()
        # Set train loss.
        train_loss = 0.0

        # Loop through X batch, y batch.
        for X_batch, y_batch in train_loader:
            # Set X batch.
            X_batch = X_batch.to(device)
            # Set y batch.
            y_batch = y_batch.to(device)

            # Clear gradients accumulated by `optimizer` before the next backward pass.
            optimizer.zero_grad()
            # Set predictions.
            predictions = model(X_batch)
            # Set loss.
            loss = criterion(predictions, y_batch)
            # Backpropagate through `loss` to calculate parameter gradients.
            loss.backward()
            # Run clip_grad_norm_.
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )
            # Apply one parameter-update step with `optimizer`.
            optimizer.step()

            # Update train loss.
            train_loss += loss.item() * len(X_batch)

        # Update train loss.
        train_loss /= len(train_loader.dataset)

        # Switch `model` to evaluation mode.
        model.eval()
        # Set val loss.
        val_loss = 0.0
        # Open the requested managed context and clean up its resources automatically.
        with torch.no_grad():
            # Loop through X batch, y batch.
            for X_batch, y_batch in val_loader:
                # Set X batch.
                X_batch = X_batch.to(device)
                # Set y batch.
                y_batch = y_batch.to(device)
                # Set predictions.
                predictions = model(X_batch)
                # Set loss.
                loss = criterion(predictions, y_batch)
                # Update val loss.
                val_loss += loss.item() * len(X_batch)

        # Update val loss.
        val_loss /= len(val_loader.dataset)

        # Check the condition.
        if params["scheduler_name"] == "ReduceLROnPlateau":
            # Update the learning rate.
            scheduler.step(val_loss)
        else:
            # Apply one parameter-update step with `scheduler`.
            scheduler.step()

        # Check the condition.
        if best_model_state is None or val_loss < best_val_loss:
            # Store the best validation loss.
            best_val_loss = val_loss
            # Store the best epoch.
            best_epoch = epoch
            # Set best model state.
            best_model_state = deepcopy(model.state_dict())
            # Set epochs without improvement.
            epochs_without_improvement = 0
        else:
            # Update epochs without improvement.
            epochs_without_improvement += 1

        # Check the condition.
        if (
            epochs_without_improvement
            >= params["early_stopping_patience"]
        ):
            # Exit the nearest loop because no further iterations are required.
            break

    # Check the condition.
    if best_model_state is None:
        # Stop with an error.
        raise ValueError(
            "No valid model state was saved. Check for NaN validation loss."
        )

    # Run load_state_dict.
    model.load_state_dict(best_model_state)
    # Switch `model` to evaluation mode.
    model.eval()

    # Set test prediction batches.
    test_prediction_batches = []
    # Open the requested managed context and clean up its resources automatically.
    with torch.no_grad():
        # Loop through X batch,.
        for X_batch, _ in test_loader:
            # Set X batch.
            X_batch = X_batch.to(device)
            # Run append.
            test_prediction_batches.append(
                model(X_batch).cpu().numpy()
            )

    # Set test pred log.
    test_pred_log = np.concatenate(
        test_prediction_batches,
        axis=0,
    )
    # Set test predictions.
    test_predictions = mh.inverse_target_transform(
        test_pred_log,
        transform="log1p",
    )

    # Set metrics.
    metrics = mh.evaluate_two_targets(
        metadata_df=fold["test_metadata"],
        y_true=fold["y_test_raw"],
        y_pred=test_predictions,
        date_col=date_col,
        id_col=id_col,
        target_labels=("Inflow", "Outflow"),
        k_values=(10, 20),
    )

    # Return the result.
    return {
        "model": model,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "test_pred_log": test_pred_log,
        "test_predictions": test_predictions,
        "metrics": metrics,
    }


## 9. Full grid search and CatBoost-style result files


In [ ]:
# Set feature set name.
feature_set_name = (
    "full_static_dynamic_node2vec_"
    "wikidata_minilm_pca"
)
# Set feature set code.
feature_set_code = "FULL"
# Set split type.
split_type = "SPCV"
# Set target transform.
target_transform = "log1p"
# Set model name.
model_name = "MLPRegressorMultiOutput"
# Set project name.
project_name = "toronto-cyclist-attractiveness-dev"

# Set results dir.
results_dir = (
    project_root
    / "notebooks/thats/bike_prepare_data/results"
)
# Run mkdir.
results_dir.mkdir(parents=True, exist_ok=True)

# Set results file.
results_file = (
    results_dir
    / "mlp_multioutput_FULL_spcv_grid_results.csv"
)
# Set fold results file.
fold_results_file = (
    results_dir
    / "mlp_multioutput_FULL_spcv_fold_results.csv"
)

# Set metric cols.
metric_cols = [
    f"{prefix}_{metric}"
    for prefix in ("Inflow", "Outflow", "Avg")
    for metric in mh.METRIC_NAMES
]


In [ ]:
# Set RUN GRID SEARCH.
RUN_GRID_SEARCH = True

# Store all results.
all_results = []
# Set all fold results.
all_fold_results = []

# Check the condition.
if RUN_GRID_SEARCH:
    # Loop through trial number, params.
    for trial_number, params in enumerate(
        parameter_combinations,
        start=1,
    ):
        # Set run name.
        run_name = (
            f"E_MLP_MULTI_{feature_set_code}_"
            f"{split_type}_trial_{trial_number:03d}"
        )

        # Run W&B only when enabled.
        if USE_WANDB:
            # Set run.
            run = wandb.init(
                project=project_name,
                name=run_name,
                tags=[
                    model_name,
                    feature_set_code,
                    split_type,
                    target_transform,
                    params["optimizer_name"],
                    params["scheduler_name"],
                ],
                config={
                    "trial_number": trial_number,
                    "run_name": run_name,
                    "feature_set_name": feature_set_name,
                    "model_name": model_name,
                    "n_spcv_folds": len(prepared_folds),
                    **params,
                },
            )
        else:
            # Set run.
            run = None

        # Set trial fold results.
        trial_fold_results = []

        # Loop through fold.
        for fold in prepared_folds:
            # Set fold output.
            fold_output = train_one_mlp_fold(
                fold=fold,
                params=params,
                device=device,
            )

            # Set fold result.
            fold_result = {
                "trial_number": trial_number,
                "run_name": run_name,
                "feature_set_name": feature_set_name,
                "feature_set_code": feature_set_code,
                "split_type": split_type,
                "target_transform": target_transform,
                "model_name": model_name,
                "fold_id": fold["fold_id"],
                "n_train_rows": len(fold["train_df"]),
                "n_val_rows": len(fold["val_df"]),
                "n_test_rows": len(fold["test_df"]),
                "n_train_stations": (
                    fold["train_df"][id_col].nunique()
                ),
                "n_val_stations": (
                    fold["val_df"][id_col].nunique()
                ),
                "n_test_stations": (
                    fold["test_df"][id_col].nunique()
                ),
                "train_groups": fold["train_groups"],
                "val_groups": fold["val_groups"],
                "test_groups": fold["test_groups"],
                "best_epoch": fold_output["best_epoch"],
                "best_val_loss": fold_output["best_val_loss"],
                **params,
                **fold_output["metrics"],
            }

            # Run append.
            trial_fold_results.append(fold_result)
            # Run append.
            all_fold_results.append(fold_result)

            # Print the result.
            print(
                f"trial={trial_number:03d} | "
                f"fold={fold['fold_id']:02d} | "
                f"best_epoch={fold_output['best_epoch']:03d} | "
                f"Avg_RMSE={fold_output['metrics']['Avg_RMSE']:.4f} | "
                f"Avg_Recall@20="
                f"{fold_output['metrics']['Avg_Recall@20']:.4f}"
            )

        # Set trial fold df.
        trial_fold_df = pd.DataFrame(trial_fold_results)

        # Set result row.
        result_row = {
            "trial_number": trial_number,
            "run_name": run_name,
            "feature_set_name": feature_set_name,
            "feature_set_code": feature_set_code,
            "split_type": split_type,
            "target_transform": target_transform,
            "model_name": model_name,
            "n_spcv_folds": len(prepared_folds),
            **params,
            **mh.aggregate_fold_metrics(
                trial_fold_df,
                metric_cols=metric_cols,
            ),
            "best_epoch_mean": (
                trial_fold_df["best_epoch"].mean()
            ),
            "best_epoch_std": (
                trial_fold_df["best_epoch"].std()
            ),
            "best_val_loss_mean": (
                trial_fold_df["best_val_loss"].mean()
            ),
            "best_val_loss_std": (
                trial_fold_df["best_val_loss"].std()
            ),
        }
        # Run append.
        all_results.append(result_row)

        # Run W&B only when enabled.
        if USE_WANDB:
            # Record the supplied metrics or metadata in `wandb`.
            wandb.log(
                {
                    f"spcv/{key}": value
                    for key, value in result_row.items()
                    if key.endswith("_mean")
                    or key.endswith("_std")
                }
            )
            # Finish and close the current `run` run.
            run.finish()

    # Set results df.
    results_df = (
        pd.DataFrame(all_results)
        .sort_values(
            by="Avg_RMSE_mean",
            ascending=True,
        )
        .reset_index(drop=True)
    )

    # Set fold results df.
    fold_results_df = pd.DataFrame(
        all_fold_results
    )

    # Run to_csv.
    results_df.to_csv(results_file, index=False)
    # Run to_csv.
    fold_results_df.to_csv(
        fold_results_file,
        index=False,
    )

    # Print the result.
    print("Best MLP multi-output configuration:")
    # Show the result.
    display(results_df.head(1))

    # Print the result.
    print("All trial results:")
    # Show the result.
    display(results_df)

    # Print the result.
    print("All fold results:")
    # Show the result.
    display(fold_results_df)


## Reuse boundary

**Kept in the CatBoost preparation notebook:** raw joins, feature construction, missing-value decisions, row-balanced spatial fold search, fold-assignment creation, and parquet saving.

**Kept in this MLP notebook:** loading the full `all_wikidata_tokens.parquet`, creating and saving `all_wikidata_minilm_embeddings.parquet`, joining MiniLM features to the MLP dataframe, dropping `wiki_items_text`, PCA configuration, MLP fold matrix creation, DataLoaders, MLP architecture, optimiser/scheduler selection, early stopping, and the MLP grid-search loop.

**Moved to `model_helper_stations_lvl.py`:** loading the shared modelling parquet files, reconstruction of saved folds, target transformations, regression/ranking/two-target evaluation, fold aggregation, random seeding, numerical/categorical preprocessing, and shared neural encoder/regression head.

**Reused later by GAT:** `all_wikidata_minilm_embeddings.parquet` is loaded directly from the graph folder and merged by `loc_id`; GAT does not recreate the embeddings.
